In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### DONE - ML #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# DONE - ML update with your username and password and CRUD Python module name

username = "aacuser"
password = "Drink@123!"


# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

# Column Header Text Formatting - ML
df.rename(columns={
    'age_upon_outcome': 'Age Upon Outcome',
    'animal_id': 'Animal ID',
    'animal_type': 'Animal Type',
    'breed': 'Breed',
    'color': 'Color',
    'date_of_birth': 'Date of Birth',
    'datetime': 'Date & Time',
    'name': 'Name',
    'outcome_subtype': 'Outcome Subtype',
    'outcome_type': 'Outcome Type',
    'sex_upon_outcome': 'Sex Upon Outcome',
}, inplace=True)

#Reorder Columns for User readability - ML
desired_order = [
    'Animal ID', 
    'Name', 
    'Animal Type', 
    'Breed', 
    'Date of Birth',
    'Color',
    'Sex Upon Outcome', 
    'Age Upon Outcome',
    'Outcome Type', 
    'Outcome Subtype',
    'Date & Time',
    'rec_num', 
    'location_lat',
    'location_long',
    'monthyear',
    'age_upon_outcome_in_weeks'
]

# Apply new order - ML
df = df[desired_order]

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('Grazioso Salvare')

#DONE - ML Add in Grazioso Salvare’s logo
image_filename = 'GSLogo.png' # replace with Grazioso Salvare logo
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#DONE - ML Place the HTML image tag in the line below into the app.layout code according to your design
#DONE - ML Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(
        html.A(
            href='https://www.snhu.edu',
            target='_blank',
            children=[
                html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '200px'})
            ]
        )
    ),
    html.Center(html.B(html.H1('Grazioso Salvare Animal Directory'))),
    html.I('Developed By: OG_Busty Stacks'),
    html.Hr(),
    #DONE - ML Premade interactive filter options in a dropdown menu format
    # Starts with 'Reset" preselected as that is the same data without filters.
    html.Div([
    dcc.Dropdown(id='filter-type', options=['Disaster Rescue or Individual Tracking', 'Mountain or Wilderness Rescue', 'Water Rescue', 'Reset'], value='Reset')
    ]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         
                        #DONE - ML: Set up the features for your interactive data table to make it user-friendly for your client
                        filter_action="native", # Filter column data mathcing user input
                        sort_action="native", # User Dash native sort function
                        sort_mode="multi", # Allow user to sort multiple columns at a time
                        # The next three lines 'pagify' the data, starting on page 1 (index 0), and limiting the user to only see 25 entries per page
                        page_action="native", 
                        page_current=0,
                        page_size=25,
                        hidden_columns=['rec_num', 'location_lat', 'location_long', 'monthyear', 'age_upon_outcome_in_weeks'], # Hiding specific columns to declutter user facing data
                        row_selectable='single', # User can select one row to use with our mapping function
                        selected_rows=[0]
        
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## DONE - ML Add code to filter interactive data table with MongoDB queries
# If/Else loop for the datatable to update based on the pre-made search queries
# There are the three specific queries provided by GS, and the final Else loop resets to original state
    if filter_type=='Disaster Rescue or Individual Tracking':
        query={
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20.0, "$lte": 300.0}
        }
        
    elif filter_type=='Mountain or Wilderness Rescue':
        query={
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
        
    elif filter_type=='Water Rescue':
        query={
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26.0, "$lte": 156.0}
        }
        
    else:
        query={}

    filtered_data=db.read(query)
    
    df=pd.DataFrame.from_records(filtered_data)
    
    df.drop(columns=['_id'],inplace=True)
    # Rename column headers to clean up look of interface
    df.rename(columns={
        'age_upon_outcome': 'Age Upon Outcome',
        'animal_id': 'Animal ID',
        'animal_type': 'Animal Type',
        'breed': 'Breed',
        'color': 'Color',
        'date_of_birth': 'Date of Birth',
        'datetime': 'Date & Time',
        'name': 'Name',
        'outcome_subtype': 'Outcome Subtype',
        'outcome_type': 'Outcome Type',
        'sex_upon_outcome': 'Sex Upon Outcome',
    }, inplace=True)
    # Reordered the columns for a more organized view of the filtered data
    desired_order=[
        'Animal ID', 
        'Name', 
        'Animal Type', 
        'Breed', 
        'Date of Birth',
        'Color',
        'Sex Upon Outcome', 
        'Age Upon Outcome',
        'Outcome Type', 
        'Outcome Subtype',
        'Date & Time',
        'rec_num', 
        'location_lat',
        'location_long',
        'monthyear',
        'age_upon_outcome_in_weeks'
    ]
    
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
# The graph updates based on the data currently displayed in the table (viewData)
def update_graphs(viewData):
    ###DONE - ML###
    # add code for chart of your choice (e.g. pie chart) #
    dff=pd.DataFrame.from_dict(viewData)
    fig=px.pie(dff, names='Breed', title='Preferred Animals by Breed', hole=0.5)
    return [
        dcc.Graph(figure=fig)    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
#DONE - ML Add in the code for your geolocation chart
def update_map(viewData, index):
    dff = pd.DataFrame.from_dict(viewData)
    if index is None:
       row = 0
    else: 
       row = index[0]
    
    # Using loc method to grab data using column names
    lat = dff.loc[row, 'location_lat']
    lon = dff.loc[row, 'location_long']
    breed = dff.loc[row, 'Breed']
    name = dff.loc[row, 'Name']
    
# Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
           center=[30.75,-97.48], zoom=10, children=[
           dl.TileLayer(id="base-layer-id"),
           dl.Marker(position=[lat, lon], children=[
               dl.Tooltip(breed),
               dl.Popup([
                   html.H1("Animal Name"),
                   html.P(name)
                ])
              ])
           ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://biologycanyon-doorenrico-3000.codio.io/proxy/8050/
